In [31]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [32]:
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")  # Importante para alguns sistemas
    
    # Desative o modo headless temporariamente para debug
    # chrome_options.add_argument("--headless")  
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

In [33]:
def extrair_dados_asma():
    driver = setup_driver()
    try:
        print("[1/5] Acessando TABNET SIH...")
        driver.get("http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/niuf.def")
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        
        print("[2/5] Configurando parâmetros...")
        # Configura Linha e Coluna
        Select(WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.NAME, "Linha"))
        )).select_by_visible_text("Capítulo CID-10")
        
        Select(WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.NAME, "Coluna"))
        )).select_by_visible_text("Ano processamento")
        
        print("[3/5] Localizando CID J45...")
        # Rola até a seção de CIDs
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        # Método alternativo para encontrar J45
        j45_element = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(translate(text(), 'J45', 'j45'), 'j45')]/preceding-sibling::input"))
        )
        driver.execute_script("arguments[0].click();", j45_element)
        
        print("[4/5] Selecionando anos...")
        # Seleciona os últimos 5 anos
        for ano in range(2020, 2025):
            try:
                year_checkbox = driver.find_element(By.XPATH, f"//input[@value='{ano}']")
                driver.execute_script("arguments[0].click();", year_checkbox)
            except Exception as e:
                print(f"Checkbox do ano {ano} não encontrado: {str(e)}")
        
        print("[5/5] Gerando relatório...")
        driver.find_element(By.XPATH, "//input[@value='Mostra']").click()
        
        # Aguarda o relatório carregar
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.ID, "tab"))
        )
        time.sleep(3)  # Tempo extra para dados carregarem
        
        # Extrai a tabela
        html = driver.page_source
        return pd.read_html(html, attrs={'id': 'tab'})[0]
        
    except Exception as e:
        print(f"Erro fatal: {str(e)}")  # Linha corrigida
        return None
    finally:
        driver.quit()

In [34]:
print("Iniciando extração...")
df_asma = extrair_dados_asma()

Iniciando extração...
[1/5] Acessando TABNET SIH...
[2/5] Configurando parâmetros...
[3/5] Localizando CID J45...
Erro fatal: Message: 
Stacktrace:
#0 0x555a7277175a <unknown>
#1 0x555a722140a0 <unknown>
#2 0x555a722659b0 <unknown>
#3 0x555a72265ba1 <unknown>
#4 0x555a722b3ea4 <unknown>
#5 0x555a7228b3cd <unknown>
#6 0x555a722b12a0 <unknown>
#7 0x555a7228b173 <unknown>
#8 0x555a72257d4b <unknown>
#9 0x555a722589b1 <unknown>
#10 0x555a7273690b <unknown>
#11 0x555a7273a80a <unknown>
#12 0x555a7271e662 <unknown>
#13 0x555a7273b394 <unknown>
#14 0x555a7270349f <unknown>
#15 0x555a7275f538 <unknown>
#16 0x555a7275f716 <unknown>
#17 0x555a727705c6 <unknown>
#18 0x7f9959d5b1f5 <unknown>



In [ ]:
if df_asma is not None:
    print("✅ Extração concluída com sucesso!")
    print(f"Total de registros: {len(df_asma)}")
    print("\nAmostra dos dados:")
    print(df_asma.head())
else:
    print("❌ Falha na extração dos dados")